In [2]:
import sys
import os
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from scipy.stats import norm
from scipy.optimize import brentq

# Add local modules
sys.path.append('.')
import BTSSlayers as btss

# Suppress warnings
warnings.filterwarnings("ignore")

In [3]:
def find_intersection(mean1, std1, mean2, std2):
    """Find intersection between two Gaussian PDFs."""
    func = lambda x: norm.pdf(x, mean1, std1) - norm.pdf(x, mean2, std2)
    a = min(mean1 - 3 * std1, mean2 - 3 * std2)
    b = max(mean1 + 3 * std1, mean2 + 3 * std2)
    
    if func(a) * func(b) > 0:
        return (mean1 + mean2) / 2
    try:
        return brentq(func, a, b)
    except ValueError:
        return (mean1 + mean2) / 2

def get_gmm_thresholds_iki(log_iki_series):
    """
    Fits 3-component GMM to LogIKI data and returns thresholds.
    Returns dictionary with gmm_iki_auto and gmm_iki_cons.
    """
    X = log_iki_series.dropna().values.reshape(-1, 1)
    # Default NaNs
    res = {'GMM_Threshold_LogIKI': np.nan, 'gmm_iki_auto': np.nan, 'gmm_iki_cons': np.nan}
    
    # Needs enough data points
    if len(X) < 5: 
        return res

    try:
        # --- 2 Component  ---
        gmm2 = GaussianMixture(n_components=2, random_state=42)
        gmm2.fit(X)
        means2 = gmm2.means_.flatten()
        stds2 = np.sqrt(gmm2.covariances_.flatten())
        order2 = np.argsort(means2)
        res['GMM_Threshold_LogIKI'] = find_intersection(means2[order2[0]], stds2[order2[0]], 
                                                        means2[order2[1]], stds2[order2[1]])

        # --- 3 Component ---
        gmm3 = GaussianMixture(n_components=3, random_state=42)
        gmm3.fit(X)
        means = gmm3.means_.flatten()
        stds = np.sqrt(gmm3.covariances_.flatten())
        order = np.argsort(means)
        
        # Automatic Threshold: Intersection of Comp 0 & 1 (Fastest & Middle)
        res['gmm_iki_auto'] = find_intersection(means[order[0]], stds[order[0]], 
                                                means[order[1]], stds[order[1]])
        # Conscious Threshold: Intersection of Comp 1 & 2 (Middle & Slowest)
        res['gmm_iki_cons'] = find_intersection(means[order[1]], stds[order[1]], 
                                                means[order[2]], stds[order[2]])
    except Exception:
        pass
        
    return res

In [4]:
def process_single_session(session_name, tables):
    """
    Computes all metrics for a SINGLE session data dictionary.
    """
    results = {'StudySession': session_name}
    
    au = tables.get('au1')
    kd = tables.get('kd')
    tt = tables.get('tt')
    
    if au is None or kd is None or tt is None or kd.empty:
        return None

    # --- 1. AU Metrics (LogPUB, LogKBI) ---
    if not au.empty and 'PUB' in au.columns and 'KBI' in au.columns:
        pub_val = au['PUB'].iloc[0]
        kbi_val = au['KBI'].iloc[0]
        results['LogPUB'] = np.log(pub_val / 3) if pub_val > 0 else np.nan
        results['LogKBI'] = np.log(kbi_val / 2) if kbi_val > 0 else np.nan
    else:
        results['LogPUB'] = np.nan
        results['LogKBI'] = np.nan

    # --- 2. Calculate Standard IKI (NextPause - NextDur) ---
    kd['NextChar'] = kd['Char'].shift(-1).fillna('_')
    kd['BI'] = (kd['Char'] + kd['NextChar']).astype(str)
    
    kd['NextType'] = kd['Type'].shift(-1).fillna(kd['Type'])
    kd['NextPause'] = kd['Pause'].shift(-1).fillna(kd['Pause'])
    kd['NextDur'] = kd['Dur'].shift(-1).fillna(kd['Dur'])
    kd['IKI'] = kd['NextPause'] - kd['NextDur']
    
    # Border Logic
    conditions = [
        kd['BI'].str.match(r"__"),     # SS
        kd['BI'].str.match(r"_[^_]"),  # SW
        kd['BI'].str.match(r"[^_]_"),  # WS
        kd['BI'].str.match(r"[^_][^_]")# WW
    ]
    choices = ['SS', 'SW', 'WS', 'WW']
    kd['Border'] = np.select(conditions, choices, default='XX')

    # --- 3. Compute Stats ---
    lang = kd['TL'].iloc[0] if 'TL' in kd.columns else 'en'
    chunk_length = 1.5 if lang in ['zh', 'ja'] else 3

    valid_mask = (kd['Type'] == 'Mins') & (kd['NextType'] == 'Mins')
    valid_rows = kd[valid_mask]

    WW = []
    BW = []
    
    if lang in ['zh', 'ja']:
        if not valid_rows.empty:
            dur_per_stroke = valid_rows['Dur'] / valid_rows['Strokes']
            counts = valid_rows['Strokes'].values
            WW = np.repeat(dur_per_stroke.values, counts.astype(int)) 
            BW = valid_rows['IKI'].values
    else:
        WW = valid_rows.loc[valid_rows['Border'] == 'WW', 'IKI'].values
        BW = valid_rows.loc[valid_rows['Border'] == 'WS', 'IKI'].values

    # Word Length (wl) calculation from TT
    chars_count = tt['TToken'].str.len().sum()
    words_count = len(tt)
    wl = chars_count / words_count if words_count > 0 else 1

    # Medians
    median_ww = np.median(WW) if len(WW) > 0 else 0
    median_bw = np.median(BW) if len(BW) > 0 else 0

    results['KBI2'] = 2 * int(median_ww)
    results['PUB3'] = 3 * int(median_bw)

    # BW_PUB2 (Quantile based on wl)
    q_target = max(0.0, min(1.0, 1 - (1 / (wl * chunk_length))))
    results['BW_PUB2'] = int(np.quantile(kd['IKI'].dropna(), q_target)) if not kd['IKI'].dropna().empty else 0
    
    results['QBW3'] = np.mean(kd['IKI'] < results['PUB3'])
    results['QWW2'] = np.mean(kd['IKI'] < results['KBI2'])
    
    # --- 4. LogIKI Calculation (Based on Time Diff) ---
    kd = kd.sort_values('Time')
    kd['IKI_TimeDiff'] = kd['Time'].diff()
    
    # Filter valid positive values for Log calculation
    valid_time_diffs = kd['IKI_TimeDiff'][kd['IKI_TimeDiff'] > 0].dropna()
    log_iki = np.log(valid_time_diffs)
    
    # Quantile_iki: Explicitly 75th percentile of LogIKI
    results['Quantile_iki'] = np.quantile(log_iki, 0.75) if not log_iki.empty else np.nan

    # --- 5. GMM Analysis (LogIKI) ---
    gmm_res = get_gmm_thresholds_iki(log_iki)
    results.update(gmm_res)

    return results

In [5]:
print("Reading session list...")
GD = pd.read_csv('sorted.gaze.clean.txt', sep="\t", dtype=None)
session_list = GD['Study-Session'].unique()

all_session_data = []

print(f"Processing {len(session_list)} sessions sequentially...")

for i, session in enumerate(session_list):
    if i % 10 == 0:
        print(f"  Processed {i}/{len(session_list)} sessions... (Memory Cleanup)")
        gc.collect()

    try:
        # Read only specific session
        tables = btss.readBTSSsessions([session], layers=['au1', 'fd', 'kd', 'tt'], verbose=0)
        
        row_metrics = process_single_session(session, tables)
        
        if row_metrics:
            all_session_data.append(row_metrics)
            
        del tables
        
    except Exception as e:
        print(f"  Error processing session {session}: {e}")
        continue

Reading session list...
Processing 490 sessions sequentially...
  Processed 0/490 sessions... (Memory Cleanup)
  Processed 10/490 sessions... (Memory Cleanup)
  Processed 20/490 sessions... (Memory Cleanup)
  Processed 30/490 sessions... (Memory Cleanup)
  Processed 40/490 sessions... (Memory Cleanup)
  Processed 50/490 sessions... (Memory Cleanup)
  Processed 60/490 sessions... (Memory Cleanup)
  Processed 70/490 sessions... (Memory Cleanup)
  Processed 80/490 sessions... (Memory Cleanup)
  Processed 90/490 sessions... (Memory Cleanup)
  Processed 100/490 sessions... (Memory Cleanup)
  Processed 110/490 sessions... (Memory Cleanup)
  Processed 120/490 sessions... (Memory Cleanup)
  Processed 130/490 sessions... (Memory Cleanup)
  Processed 140/490 sessions... (Memory Cleanup)
  Processed 150/490 sessions... (Memory Cleanup)
  Processed 160/490 sessions... (Memory Cleanup)
  Processed 170/490 sessions... (Memory Cleanup)
  Processed 180/490 sessions... (Memory Cleanup)
  Processed 190/

In [6]:
print("Creating final DataFrame...")
final_df = pd.DataFrame(all_session_data)

cols_to_save = [
    "StudySession", "LogKBI", "LogPUB", 
    "GMM_Threshold_LogIKI", "gmm_iki_auto", "gmm_iki_cons", 
    "Quantile_iki", "QWW2", "QBW3", "BW_PUB2"
]

for c in cols_to_save:
    if c not in final_df.columns:
        final_df[c] = np.nan

final_metrics = final_df[cols_to_save]

Creating final DataFrame...


In [8]:
final_metrics[['LogPUB','LogKBI','GMM_Threshold_LogIKI','gmm_iki_cons','gmm_iki_auto','Quantile_iki','QWW2','QBW3','BW_PUB2']].corr()

,LogPUB,LogKBI,GMM_Threshold_LogIKI,gmm_iki_cons,gmm_iki_auto,Quantile_iki,QWW2,QBW3,BW_PUB2
LogPUB,1.000000,0.800586,0.046285,0.613597,-0.027028,0.860254,-0.370777,-0.441764,0.750206
LogKBI,0.800586,1.000000,0.003830,0.565944,-0.080606,0.929476,-0.074924,-0.305363,0.628018
GMM_Threshold_LogIKI,0.046285,0.003830,1.000000,0.237387,0.298464,0.064274,-0.185238,0.020296,0.053070
gmm_iki_cons,0.613597,0.565944,0.237387,1.000000,0.336767,0.640018,-0.342415,-0.313131,0.583013
gmm_iki_auto,-0.027028,-0.080606,0.298464,0.336767,1.000000,-0.048625,-0.032929,0.056211,0.008453
Quantile_iki,0.860254,0.929476,0.064274,0.640018,-0.048625,1.000000,-0.397836,-0.390168,0.669128
QWW2,-0.370777,-0.074924,-0.185238,-0.342415,-0.032929,-0.397836,1.000000,0.302350,-0.261538
QBW3,-0.441764,-0.305363,0.020296,-0.313131,0.056211,-0.390168,0.302350,1.000000,-0.473910
BW_PUB2,0.750206,0.628018,0.053070,0.583013,0.008453,0.669128,-0.261538,-0.473910,1.000000
